# Week 4: PLS vs Ridge / LR (LORO)

Supervised compression of AlphaEarth `A00`–`A63` with **Partial Least Squares**, compared to the week-4 balanced pure-cell baselines on the same LORO folds.

AE bands are **not orthogonal** (mean pairwise |corr| ≈ 0.22; effective rank ≈ 10). PLS scores **are** orthogonal by construction — the right place to look for a minimal set of important dimensions.

**Data.** `datasets/week4_pixel_gt_balanced/pixels.parquet` — 1,751,814 pure RES/NRES cells (1:1 per metro).

**Models**
| Family | What |
|---|---|
| `baseline_lr` / `baseline_ridge` | StandardScaler + LR / Ridge on all 64 bands |
| `pls_da_lr` | PLS on binary \(y\) → LR on scores (PLS-DA) |
| `pls_frac` | PLSRegression \(X \to\) `(frac_res, frac_nres, frac_empty)` |
| `pls_frac_lr` | PLS on fractions → LR on scores |
| `pca_lr` | PCA → LR (unsupervised control) |

Coarse sweep `n_components ∈ [2, 4, 8, 12, 16, 24, 32]`; fine PLS-DA sweep `k=1..20` for the minimum orthogonal set. Backend: **gpu** · 8 GPUs / 80 CPUs · 236.6 s.

```bash
.venv/bin/python -u scripts/train_week4_pls_loro.py --backend gpu
.venv/bin/python -u scripts/fine_week4_pls_k_sweep.py
.venv/bin/python -u scripts/build_week4_pls_notebook.py
```


## Headline

| Model | LORO macro-F1 / MAE | OOF |
|---|---:|---:|
| Baseline LR (64-D) | F1 0.779 ± 0.085 | 0.805 |
| PLS-DA → LR (k=32) | F1 0.775 ± 0.088 | 0.804 |
| PLS frac → LR (k=24) | F1 0.775 ± 0.084 | — |
| PCA → LR (k=32) | F1 0.759 ± 0.084 | — |
| Baseline Ridge | MAE 0.263 ± 0.027 | 0.254 |
| PLS frac (k=12) | MAE 0.260 ± 0.028 | 0.253 |

Δ OOF F1 (PLS-DA − baseline) = **-0.0013** ·
Δ OOF MAE (PLS − Ridge) = **-0.0008**


![F1 vs n_components](../../outputs/week4_pls_loro/figures/f1_vs_ncomp.png)

![MAE vs n_components](../../outputs/week4_pls_loro/figures/mae_vs_ncomp.png)

![Best-family F1](../../outputs/week4_pls_loro/figures/best_f1_compare.png)

![Best MAE](../../outputs/week4_pls_loro/figures/best_mae_compare.png)

![Per-metro PLS vs baseline](../../outputs/week4_pls_loro/figures/loro_f1_pls_vs_baseline.png)

![OOF confusion](../../outputs/week4_pls_loro/figures/oof_confusion_compare.png)


## Do RES / NRES form two clusters in $(t_1, t_2)$?

Yes — mostly along **$t_1$**. Orange = non-residential (left), blue = residential (right). The decision boundary is nearly vertical, so $t_2$ adds little.

| Probe on pooled OOF scores | macro-F1 |
|---|---:|
| 1D LR on $t_1$ only | 0.778 |
| 2D LR on $(t_1, t_2)$ | 0.783 |

These pooled-OOF numbers are descriptive (refit on all metros’ OOF scores). Honest LORO at k=2 is lower (~0.74 in the fine sweep) because fold transfer is harder than a global 2D cut.

```bash
.venv/bin/python -u scripts/plot_week4_pls_t1_t2.py
```

![Global t1 vs t2](../../outputs/week4_pls_loro/figures/pls_t1_t2_global.png)

![Marginals](../../outputs/week4_pls_loro/figures/pls_t1_t2_marginals.png)

![Per metro](../../outputs/week4_pls_loro/figures/pls_t1_t2_by_region.png)

Clear two-blob structure in Phoenix / Atlanta / Denver / LA / Seattle; more overlap in NYC / SF / Chicago — same pattern as the week-4 UMAP hard cities.


## Load outputs


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("../..").resolve()
OUT = ROOT / "outputs" / "week4_pls_loro"
summary = json.loads((OUT / "summary.json").read_text())
sweep = pd.read_csv(OUT / "sweep_summary.csv")
metrics = pd.read_csv(OUT / "loro_metrics.csv")
fine = json.loads((OUT / "fine_k_sweep.json").read_text())
fine_df = pd.read_csv(OUT / "fine_k_sweep.csv")
print("baseline LORO F1:", round(fine["baseline_loro_f1"], 4))
print("min-k criteria:", json.dumps(fine["min_k_criteria"], indent=2))
print("cov cumulants: 90%@k=%s  95%@k=%s  99%@k=%s" % (
    fine["cov_cum_k90"], fine["cov_cum_k95"], fine["cov_cum_k99"]))


## Minimum orthogonal dimensions (fine PLS-DA sweep)

PLS-DA scores are **orthogonal**. Fine LORO sweep over `k = 1..20` on the same balanced pure cells (8 GPUs).

```bash
.venv/bin/python -u scripts/fine_week4_pls_k_sweep.py
```

### Covariance with \(y\) (signal concentration)

Almost all supervised covariance sits in the first few scores:

| Cumulative cov²(t, y) | k |
|---|---:|
| ~90% | **1** |
| ~95% | **2** |
| ~99% | **5** |

### Classifier parity vs 64-D LR (LORO F1 = 0.7792)

| Criterion | Min k | F1 | gap |
|---|---:|---:|---:|
| within 0.02 F1 | **9** | 0.7612 | 0.0180 |
| within 0.01 F1 | **16** | 0.7708 | 0.0084 |
| within 2% relative | **11** | 0.7673 | 0.0119 |
| within 1% relative | **17** | 0.7723 | 0.0069 |

### Selected k on the fine curve

| k | LORO F1 | gap to 64-D | rel gap |
|---:|---:|---:|---:|
| 1 | 0.735 | 0.0444 | 0.0569 |
| 2 | 0.738 | 0.0415 | 0.0532 |
| 5 | 0.749 | 0.0302 | 0.0388 |
| 9 | 0.761 | 0.0180 | 0.0231 |
| 11 | 0.767 | 0.0119 | 0.0153 |
| 12 | 0.765 | 0.0137 | 0.0176 |
| 16 | 0.771 | 0.0084 | 0.0108 |
| 17 | 0.772 | 0.0069 | 0.0088 |
| 20 | 0.776 | 0.0029 | 0.0037 |
| 64 | 0.779 | 0.0000 | 0.0000 |

**Recommendation**
- Absolute minimum for the residential *signal*: **k = 5** (99% of cov with \(y\)).
- Best compression that still races LR: **k = 9–12**.
- Near-parity with full 64-D LR: **k = 16**.

![Fine k minimum](../../outputs/week4_pls_loro/figures/fine_k_minimum.png)


In [ ]:
fine_df


### Binary heads only (coarse sweep)


In [ ]:
bin_models = ["baseline_lr", "pls_da_lr", "pls_frac_lr", "pca_lr"]
sweep.loc[sweep.model.isin(bin_models), [
    "model", "n_components", "loro_macro_f1_mean", "loro_macro_f1_std"
]].sort_values(["model", "n_components"])


### Fraction heads (coarse sweep)


In [ ]:
frac_models = ["baseline_ridge", "pls_frac"]
sweep.loc[sweep.model.isin(frac_models), [
    "model", "n_components", "loro_mae_mean", "loro_mae_std"
]].sort_values(["model", "n_components"])


## Takeaways

1. **AE bands are collinear** (not orthogonal; effective rank ~10). PLS scores *are* orthogonal — use those as the compressed features.
2. **PLS ≈ full LR / Ridge** at enough components (Δ OOF F1 ≈ −0.001; fraction MAE slightly better at k=12). PCA at the same width lags → the gain is supervised, not just dim reduction.
3. **Minimum orthogonal set**
   - signal (99% cov with \(y\)): **k = 5**
   - race LR (within 0.02 F1): **k = 9–12**
   - near-parity (within 0.01 F1): **k = 16**
4. Artifacts under `outputs/week4_pls_loro/`: OOF preds, best-k PLS scores (`oof_pls_da_scores_k32.npy`), fold checkpoints (loadings/rotations), coarse + fine sweeps, figures.


In [ ]:
print("paths:", json.dumps(summary.get("paths", {}), indent=2))
print("backend:", summary.get("backend"), "gpus:", summary.get("n_gpus"), "cpus:", summary.get("n_cpus"))
print("elapsed_s:", summary.get("elapsed_s"))
if (OUT / "fine_k_sweep.json").exists():
    print("fine_k_sweep:", OUT / "fine_k_sweep.json")
    print("fine_k_figure:", OUT / "figures" / "fine_k_minimum.png")
